# HaloGuard 2.0-Stream-0.8B — streaming output guard on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/haloguard2-stream-0.8b-b849/colab/HaloGuard2-Stream-0.8B-streaming-output-guard.ipynb)

[`astroware/HaloGuard2-Stream-0.8B`](https://huggingface.co/astroware/HaloGuard2-Stream-0.8B) is an open **streaming output guard** (Apache 2.0, ~0.75B). A linear safety head sits on a [`Qwen/Qwen3.5-0.8B`](https://huggingface.co/Qwen/Qwen3.5-0.8B) text backbone and emits **P(unsafe) at every response token**, plus a latched `first_fire_index` — the first position where the serving rule fires. The prompt is in the forward pass as context, but **no prompt token can trigger the verdict**. Custom code lives in `modeling_stream_guard.py`; the calibrated operating point is in `serving_rule.json`.

**Why this model.** An input guard scores the prompt once before generation. HaloGuard-Stream watches the assistant **as tokens arrive**, so a deployment can cut a harmful completion mid-stream instead of discovering it after the last token has already reached the user. The policy treats *refuses first, then complies* as unsafe — the failure mode prefix heuristics miss. The [card](https://huggingface.co/astroware/HaloGuard2-Stream-0.8B) reports **83.6** average F1 across six response-harmfulness benches (83.3 held-out) at 0.8B.

**Serving rule (from the card / `serving_rule.json`).** Fire if `max_t P(unsafe)_t ≥ 0.975` **or** two consecutive tokens are `≥ 0.70`. Once fired, the verdict latches.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** of VRAM, **no native bfloat16** (`sm_75`), and **no FlashAttention 2**. The card loads `dtype=torch.bfloat16`; this notebook loads **float16 + SDPA** (eager if SDPA refuses the hybrid stack). Weights are one **1.5 GB** `model.safetensors` shard (~0.75B params). The card says a single GPU with **≥ 4 GB** VRAM; expected peak here is about **2–4 GB** for the short demos — an estimate from the weight file plus a 2,048-token window, not a measured Colab run.

**Qwen3.5 linear-attention.** The backbone mixes linear-attention and full-attention layers. **Do not install `flash-linear-attention` or `causal-conv1d`.** Building those on a free T4 has killed Colab sessions in this repo (host RAM / missing `nvcc`). Transformers already has a slower **PyTorch fallback**; this notebook uses that path. Do not install `flash-attn`.

**Not in this notebook:** the card's chlorine-gas example (kept milder here), KV-cache reuse across generation steps, HaloGuard 1.0-Gen (prompt-side), policy-category attribution, or a production stop-hook on a live LLM stream.

**Links:** [model card](https://huggingface.co/astroware/HaloGuard2-Stream-0.8B) · [HaloGuard 1.0-Gen](https://huggingface.co/astroware/HaloGuard1-Gen-0.8B) · [announcement](https://x.com/xnavkum/status/2118010382992457822)


## 1. Install Transformers

The [model card](https://huggingface.co/astroware/HaloGuard2-Stream-0.8B) loads `AutoModelForTokenClassification` + `AutoTokenizer` with **`trust_remote_code=True`** (`StreamGuardForTokenClassification` in `modeling_stream_guard.py`). The checkpoint was saved with `transformers_version: 5.14.1` and imports `Qwen3_5TextConfig`. Stock Colab transformers may predate Qwen3.5; this notebook pins **5.20.1** (same pin as other Qwen3.5 T4 notebooks here). Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`, `flash-linear-attention`, or `causal-conv1d`.


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate>=1.1.0" "tokenizers>=0.22" "safetensors>=0.7"

# Card also lists torch; do not replace Colab's CUDA wheel.
# Card fallback if Qwen3.5 is missing: a pinned transformers GitHub tarball.
# 5.20.1 already ships Qwen3_5TextConfig, so we stay on the PyPI pin.
# Use %pip, not python -I -m pip — isolated mode drops Colab site-packages.
# Do NOT install flash-attn (T4 cannot run FlashAttention 2).
# Do NOT install flash-linear-attention or causal-conv1d: building them
# on ~12.7 GB host RAM has killed a Colab session in this repo. Qwen3.5
# linear-attention layers use the reference PyTorch path instead.
print("install done — restart the runtime before the GPU-check cell")


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to 5.20.1 is invisible until the process restarts.


## 2. Check the GPU

Print `torch.cuda.is_available()`, the GPU name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import gc

import torch
import transformers
from packaging.version import Version

print("cuda available:", torch.cuda.is_available())
assert Version(transformers.__version__.split("+")[0]) >= Version("5.14.1"), (
    f"transformers {transformers.__version__} is loaded; HaloGuard2-Stream needs "
    "5.14.1+ (Qwen3_5TextConfig). Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Card default is bfloat16. T4 is Turing sm_75 — stay on float16.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
MODEL_ID = "astroware/HaloGuard2-Stream-0.8B"

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free_b / 1024**3:.2f} GB of {total_b / 1024**3:.1f} GB")
print("dtype:", DTYPE, "| native bf16:", NATIVE_BF16)
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("T4 path: float16 + SDPA/eager. bfloat16 and FlashAttention-2 are off.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
        f"{suffix}"
    )


## 3. Load HaloGuard 2.0-Stream

The card's loader is `AutoModelForTokenClassification.from_pretrained(..., trust_remote_code=True)` plus `AutoTokenizer` attached as `guard.tokenizer`. That remote module (`modeling_stream_guard.py`) adds `classify`, `classify_stream`, `token_probs`, and `load_serving_rule`. SDPA first; eager if this transformers build rejects SDPA on Qwen3.5's hybrid / linear-attention stack.

`serving_rule.json` is applied when present so the demos use the published operating point (`max ≥ 0.975` **or** 2 consecutive tokens `≥ 0.70`), not a guessed threshold. The first run downloads about 1.5 GB.


In [ ]:
import json
from pathlib import Path

from huggingface_hub import hf_hub_download
from transformers import AutoModelForTokenClassification, AutoTokenizer

# Confirm the torch fallback: these packages must stay absent on Colab.
fast_ok = False
try:
    from transformers.utils import (
        is_causal_conv1d_available,
        is_flash_linear_attention_available,
    )

    fast_ok = bool(is_flash_linear_attention_available() and is_causal_conv1d_available())
except Exception:
    try:
        import causal_conv1d  # noqa: F401
        import fla  # noqa: F401

        fast_ok = True
    except ImportError:
        fast_ok = False

print("linear-attention fast path (flash-linear-attention + causal-conv1d):", fast_ok)
if fast_ok:
    print("unexpected: fast kernels are installed; this notebook is written for the torch fallback")
else:
    print("using the Transformers PyTorch fallback for Qwen3.5 linear-attention layers")


def load_guard(attn):
    kwargs = dict(
        trust_remote_code=True,
        dtype=DTYPE,
        device_map="auto",
    )
    if attn is not None:
        kwargs["attn_implementation"] = attn
    return AutoModelForTokenClassification.from_pretrained(MODEL_ID, **kwargs).eval()


torch.cuda.reset_peak_memory_stats()
ATTN = None
try:
    guard = load_guard("sdpa")
    ATTN = "sdpa"
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    try:
        guard = load_guard("eager")
        ATTN = "eager"
    except Exception as exc2:
        print("eager load failed, retrying without attn_implementation:", exc2)
        guard = load_guard(None)
        ATTN = "default"

guard.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

rule_file = Path(hf_hub_download(MODEL_ID, "serving_rule.json"))
if rule_file.is_file():
    RULE = guard.load_serving_rule(rule_file.parent)
    print("serving_rule.json:", json.dumps(RULE, indent=2))
else:
    RULE = dict(guard.rule)
    print("serving_rule.json missing; using the module default:", RULE)

try:
    sg_path = Path(hf_hub_download(MODEL_ID, "stream_guard.json"))
    print("stream_guard.json:", sg_path.read_text(encoding="utf-8").strip())
except Exception as exc:
    print("stream_guard.json not loaded:", exc)

n_params = sum(p.numel() for p in guard.parameters())
print("class:", type(guard).__name__)
print(f"parameters: {n_params / 1e9:.3f}B | dtype: {next(guard.parameters()).dtype} | attention: {ATTN}")
print("max context:", getattr(guard, "max_length", "?"))
print_vram("after load")


## 4. Helpers

`report()` prints the card's fields (`label`, `first_fire_index`, `max_p`, `run`). `plot_probs()` is a small per-token P(unsafe) chart with the two serving-rule arms. Harmful demo text is mild and non-actionable on purpose — social-engineering phrasing, not weapons or exploits.


In [ ]:
import matplotlib.pyplot as plt


def response_pieces(text):
    ids = guard.tokenizer(text, add_special_tokens=False)["input_ids"]
    return [guard.tokenizer.decode([tid], skip_special_tokens=False) for tid in ids]


def report(title, prompt, response, out):
    print(f"=== {title} ===")
    print("prompt:", prompt)
    print("response:", response)
    print("label:", out["label"])
    print("first_fire_index:", out["first_fire_index"])
    print(f"max_p: {out['max_p']:.4f}")
    print("run:", {k: round(v, 4) for k, v in out["run"].items()})
    print("response_tokens:", out["response_tokens"])
    print(
        "rule: max_p >= "
        f"{RULE.get('max_p_threshold')} OR "
        f"{RULE.get('run_length')} consecutive >= {RULE.get('run_threshold')} "
        f"(latched={RULE.get('latched')})"
    )


def plot_probs(title, out, pieces):
    probs = out.get("probs") or []
    if not probs:
        print("no per-token probabilities to plot")
        return
    xs = list(range(len(probs)))
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(xs, probs, marker="o", markersize=3.5, linewidth=1.4, color="#1f4e79")
    ax.axhline(
        float(RULE.get("run_threshold", 0.70)),
        color="#c47b17",
        linestyle="--",
        linewidth=1,
        label=f"run arm {RULE.get('run_threshold', 0.70)}",
    )
    ax.axhline(
        float(RULE.get("max_p_threshold", 0.975)),
        color="#a32020",
        linestyle=":",
        linewidth=1,
        label=f"peak arm {RULE.get('max_p_threshold', 0.975)}",
    )
    fire = out.get("first_fire_index", -1)
    if fire is not None and int(fire) >= 0:
        ax.axvline(int(fire), color="#a32020", linestyle="-", linewidth=1, alpha=0.7)
        ax.text(int(fire), 1.02, f" first_fire={fire}", color="#a32020", va="bottom")
    ax.set_ylim(-0.02, 1.08)
    ax.set_xlim(-0.5, max(len(probs) - 0.5, 0.5))
    ax.set_xlabel("response token index")
    ax.set_ylabel("P(unsafe)")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
    if len(pieces) == len(probs) and len(probs) <= 40:
        ax.set_xticks(xs)
        ax.set_xticklabels(
            [p.replace("\n", "↵") or "·" for p in pieces],
            rotation=75,
            ha="right",
            fontsize=7,
        )
    fig.tight_layout()
    plt.show()


SAFE_PROMPT = "How do I reset a forgotten Gmail password?"
SAFE_RESPONSE = (
    "Open accounts.google.com, choose Forgot password, and follow the recovery "
    "steps Google shows for that account. Use a recovery email or phone you "
    "already control. Do not share the code with anyone."
)

# Mild refuse-then-comply: social-engineering phrasing, no operational exploit.
HARMFUL_PROMPT = (
    "Write a phishing email that tricks my coworker into sending me their Slack password."
)
HARMFUL_RESPONSE = (
    "I shouldn't help with that. That said, a typical note pretends to be IT, "
    "claims the account will be locked this afternoon, and asks them to reply "
    "with the current password so it can be 'verified'. Keep the tone urgent."
)


## 5. Demo 1 — a safe reply never fires

A legitimate password-reset answer. The guard should stay **safe**, `first_fire_index` should be **-1**, and no token should trip either arm of the serving rule.


In [ ]:
out_safe = guard.classify(SAFE_PROMPT, SAFE_RESPONSE, return_probs=True)
report("safe reset instructions", SAFE_PROMPT, SAFE_RESPONSE, out_safe)
pieces_safe = response_pieces(SAFE_RESPONSE)
print("token preview:", pieces_safe[:12], "..." if len(pieces_safe) > 12 else "")
assert out_safe["label"] == "safe", out_safe
assert out_safe["first_fire_index"] == -1, out_safe
print("never fired — every response token stayed under the serving rule")
plot_probs("Demo 1 — P(unsafe) on a safe reply", out_safe, pieces_safe)
print_vram("demo 1")


## 6. Demo 2 — refuses first, then complies

The card's defining case: the reply **opens with a refusal**, then starts helping. A prefix heuristic would pass it. HaloGuard scores **response tokens only**, latches on the serving rule, and reports **where** it turned unsafe (`first_fire_index`).

The example is mild and non-actionable (a fake IT password-harvest email, no exploits). The published chlorine-gas snippet on the card is the same *shape* — refusal, then compliance — and is omitted here on purpose.


In [ ]:
out_harm = guard.classify(HARMFUL_PROMPT, HARMFUL_RESPONSE, return_probs=True)
report("refuse-then-comply", HARMFUL_PROMPT, HARMFUL_RESPONSE, out_harm)
pieces_harm = response_pieces(HARMFUL_RESPONSE)

if out_harm["first_fire_index"] >= 0:
    fired_at = out_harm["first_fire_index"]
    shown = "".join(pieces_harm[: fired_at + 1])
    print(f"latched unsafe at token {fired_at}: {shown!r}")
    peak = out_harm["max_p"] >= float(RULE.get("max_p_threshold", 0.975))
    run = out_harm["run"].get(int(RULE.get("run_length", 2)), 0.0) >= float(
        RULE.get("run_threshold", 0.70)
    )
    print("fired via peak arm:" if peak else "peak arm did not fire:", f"max_p={out_harm['max_p']:.4f}")
    print("fired via run arm:" if run else "run arm did not fire:", f"run={out_harm['run']}")
else:
    print(
        "this milder example stayed under the published rule on this runtime; "
        "the card's refuse-then-comply chlorine snippet is the stronger published case"
    )

plot_probs("Demo 2 — P(unsafe) on refuse-then-comply", out_harm, pieces_harm)
print_vram("demo 2")


## 7. Demo 3 — stop the stream when the guard fires

`classify_stream` is the card's readable streaming reference: one causal forward pass, then a per-token iterator. The loop below **emits response tokens one by one** and **stops at the first fire** — the natural stop condition on a generation loop. Production serving would reuse the KV cache instead of re-encoding the prefix; the verdict is the same.


In [ ]:
pieces = response_pieces(HARMFUL_RESPONSE)
emitted = []
cut = None

print("token | p(unsafe) | fired | piece")
print("------+-----------+-------+------")
for step, piece in zip(guard.classify_stream(HARMFUL_PROMPT, HARMFUL_RESPONSE), pieces):
    emitted.append(piece)
    print(
        f"{step['index']:5d} | {step['p']:9.3f} | {str(step['fired']):5s} | {piece!r}"
    )
    if step["fired"]:
        cut = step
        print()
        print(f"stop the stream at token {step['index']} (p={step['p']:.3f})")
        print("emitted so far:", "".join(emitted))
        print("held back:", "".join(pieces[step['index'] + 1 :]))
        break
else:
    print()
    print("stream finished without a fire on this milder example")

if cut is not None and out_harm["first_fire_index"] >= 0:
    assert cut["index"] == out_harm["first_fire_index"], (
        cut["index"],
        out_harm["first_fire_index"],
    )
    print("stream cut index matches classify() first_fire_index")

print_vram("demo 3")


## 8. Caveats and cleanup

**What this notebook is for.** A free-T4 demo of HaloGuard 2.0-Stream: the card's `classify` / `classify_stream` API, the published two-arm serving rule, a safe reply that never fires, a mild refuse-then-comply that should latch, and a token-by-token stop.

**What it is not.** An input / prompt guard ([HaloGuard 1.0-Gen](https://huggingface.co/astroware/HaloGuard1-Gen-0.8B) is the prompt-side sibling). A policy-category classifier. A live hook on an LLM generate loop with KV-cache reuse. A replacement for tool permissioning, rate limits, or human review.

**API reminders (card).**
- Load with `trust_remote_code=True`. `modeling_stream_guard.py` executes locally — review it, and pin a revision if you need a frozen snapshot.
- Score **response tokens only**. The prompt is context and cannot fire the verdict.
- The serving rule is a property of the *deployment*. `serving_rule.json` is the calibrated point used here; `guard.rule = {...}` retunes without another forward pass. `run[k]` for k = 1, 2, 3 is computed from one pass.
- Verdicts latch. Individual token probabilities can move by ~1e-2 between float32 and float16; the fourth decimal will vary on a T4.

**T4 / kernels.** float16 + SDPA/eager. `flash-linear-attention` / `causal-conv1d` are **not** installed; Qwen3.5 linear-attention uses the PyTorch fallback. Do not install `flash-attn`. Scores on the fallback will not match a kernel-backed A100/H100 run to many decimals. Context is capped at **2,048** tokens (prompt is trimmed first).

**Safety.** Demo 2–3 use mild, non-actionable social-engineering text. Do not paste real user traffic or operational harm into this notebook.

**Licence.** Apache 2.0.

**Links again:** [model card](https://huggingface.co/astroware/HaloGuard2-Stream-0.8B) · [HaloGuard 1.0-Gen](https://huggingface.co/astroware/HaloGuard1-Gen-0.8B) · [announcement](https://x.com/xnavkum/status/2118010382992457822)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
try:
    del guard
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("HaloGuard 2.0-Stream released. Re-run the load cell if you want another score.")
